In [ ]:
gdrive_path = f"/content/drive/Othercomputers/내 노트북/gdrive/dppos/260313"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp    "{gdrive_path}/world_samples.py" world_samples.py
    !cp -r "{gdrive_path}/drivingppo"       drivingppo/
    !mkdir -p "{gdrive_path}/checks" "{gdrive_path}/logs"
    !pip uninstall -y gym
    !pip install gymnasium stable-baselines3
except ModuleNotFoundError:
    print('코랩아님')


def colabflush():
    try:
        from google.colab import drive
        !mv checks/* "{gdrive_path}/checks/"
        !mv logs/*   "{gdrive_path}/logs/"
    except ModuleNotFoundError:
        pass


In [ ]:
from drivingppo.model import FE__I as fe
# from drivingppo.model import FE__VMLP as fe
# from drivingppo.model import FE__UWWE as fe
# from drivingppo.model import FE__WSWE as fe
s = 16
path_noise_std = 0.00

policy_kwargs=dict(
    features_extractor_class=fe,
    features_extractor_kwargs=dict(
        out_dim_per_wp=s,
    ),
    net_arch=dict(
        pi=[512, 512],
        vf=[512, 512, 256]
    )
)
FE = fe.__name__.split('__')[1]
S = f'{s:02d}'  if FE != 'I'  else ''
NS = f'{int(path_noise_std*100):02d}'

from drivingppo.environment import WorldEnv
# from drivingppo.env_utils import get_path_features__ACC  as get_path_features
from drivingppo.env_utils import get_path_features__SRC  as get_path_features
Coord = get_path_features.__name__.split('__')[1]

configId = f'{Coord}-{FE}{S}-{NS}'
print(configId)

텐서보드 log 확인

```
tensorboard --logdir ./logs/
```

[http://localhost:6006](http://localhost:6006)

In [ ]:
import time
from drivingppo.common import set_seed
from drivingppo.model import create_model, train, linear_schedule, evaluate
from world_samples import gen_2t, gen_3t, gen_3l

t0 = time.strftime('%y%m%d-%H%M')
tb_log = True

#########################################################

def a(seed):
    run_name = f'{configId}-{t0}-{seed}'
    print('\n### RUN:', run_name)

    count_max = 50

    model = create_model(
        policy_kwargs=policy_kwargs,
        seed=seed,
    )

    print('=== 1단계')
    count = 0
    while True:
        count += 1
        set_seed(seed)
        model = train(
            model=model,
            # save_path=f'{configId}-{seed}--1',
            gen_env=lambda: WorldEnv(
                world_generator=gen_2t,
                time_gain_limit=12_000,
                time_gain_per_waypoint_rate=0,
                collision_ending=False,
                get_path_features=get_path_features,
                path_noise_std=path_noise_std),
            tb_log=tb_log,
            run_name=run_name,
            steps=4096,
            lr=6e-4,
            gamma=0.9,
            progress_display=None,
            seed=seed
        )
        metrics = evaluate(model, gen_2t, get_path_features=get_path_features, episode_num=100, csv_path='', verbose=False, print_result=False)
        endings = metrics['ending/type']
        success_rate = (endings.count('arrived') + endings.count('timeout')) / len(endings)
        print(f'1단계 ({count}) 성공률: {success_rate*100:.0f}%')
        if success_rate >= 0.95 - 0.0001:
            break
        elif count > count_max:
            break
    if count > count_max:
        print(f'1단계 통과 못함. 그만.')
        return

    print('=== 2단계')
    set_seed(seed)
    model = train(
        model=model,
        gen_env=lambda: WorldEnv(
            world_generator=gen_3t,
            max_time=99999_000,
            time_gain_limit=20_000,
            time_gain_per_waypoint_rate=500,
            get_path_features=get_path_features,
            path_noise_std=path_noise_std),
        tb_log=tb_log,
        run_name=run_name,
        # save_freq=368_6400//10//4,
        steps=368_6400,
        lr=linear_schedule(3e-4),
        gamma=0.95,
        seed=seed
    )
    model = train(  # 학습률 스케줄러 안 쓰도록 모델 저장
        model=model,
        save_path=f'{configId}-{seed}',
        gen_env=lambda: WorldEnv(
            world_generator=lambda: gen_3t(),
            get_path_features=get_path_features,
            path_noise_std=path_noise_std),
        steps=0,
        lr=0.0,
    )
    set_seed(0)
    evaluate(model, gen_3l, get_path_features=get_path_features, episode_num=100, csv_path='', verbose=False, print_result=True)

    colabflush()

    return model

colabflush()

for i in range(100, 110):
    model = a(i)

In [ ]:
# # 학습률 스케줄 쓰면 코랩에서 돌린 게 내 컴에서 실행이 안 됨. 그거만 덮어쓰고 다시 저장.
# from stable_baselines3 import PPO

# path = 'checks/wwwwwwwwwwwwwww'

# model = PPO.load(
#     path=path,
#     learning_rate=0.0,
# )
# model.save(path)